# Rabi Calibration — CGCS / Triplet-Phase-Lock Starter Notebook

**Repository:** `quantum-hardware-company`  
**Directory:** `calibration/rabi/`  

This notebook initializes the calibration layer with a minimal Rabi oscillation workflow:

1. Generate synthetic Rabi calibration data.
2. Fit a Rabi probability model.
3. Analyze residual structure.
4. Compute a CGCS / Triplet-Phase-Lock metric using cosine similarity.

## Principle

Calibration is where hardware becomes real.

Control software should follow measured system behavior, not idealized assumptions.

## Rabi model

We use the calibration primitive:

$$
P(t)=A\sin^2\left(\frac{\Omega t}{2}\right)+B
$$

where:

- $A$ is contrast / amplitude,
- $\Omega$ is Rabi frequency,
- $B$ is offset / readout background.

In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    """Rabi oscillation probability model."""
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, t, y, p0):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, t, y, p0=p0, maxfev=10000)
    return params, cov


def residuals(model, t, y, params):
    """Observed minus fitted values."""
    return y - model(t, *params)


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed signal and fitted model."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1/np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold

## Generate synthetic calibration data

This cell stands in for experimental readout data. Later, replace this block with a loader for real calibration data.

In [ ]:
# Time axis, arbitrary units
n_points = 220
t = np.linspace(0, 10, n_points)

# Ground-truth synthetic parameters
true_A = 0.88
true_Omega = 2.45
true_B = 0.06
true_params = [true_A, true_Omega, true_B]

# Noise model: measurement noise + small structured drift component
measurement_noise = 0.045 * np.random.randn(n_points)
structured_drift = 0.018 * np.sin(0.35 * t + 0.5)

y_clean = rabi_model(t, *true_params)
y_obs = y_clean + measurement_noise + structured_drift

# Probability clipping simulates bounded population readout
y_obs = np.clip(y_obs, 0, 1)

print("True parameters:")
print(f"A     = {true_A:.4f}")
print(f"Omega = {true_Omega:.4f}")
print(f"B     = {true_B:.4f}")

## Fit Rabi model

In [ ]:
initial_guess = [0.8, 2.2, 0.05]
params, cov = fit_model(rabi_model, t, y_obs, p0=initial_guess)
A_fit, Omega_fit, B_fit = params
param_std = np.sqrt(np.diag(cov))
y_fit = rabi_model(t, *params)

print("Fit parameters:")
print(f"A     = {A_fit:.4f} ± {param_std[0]:.4f}")
print(f"Omega = {Omega_fit:.4f} ± {param_std[1]:.4f}")
print(f"B     = {B_fit:.4f} ± {param_std[2]:.4f}")

## Plot data and fit

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(t, y_obs, s=18, label="observed calibration data")
plt.plot(t, y_fit, linewidth=2, label="Rabi fit")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title("Rabi calibration: measured response and fitted model")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rabi_01_fit.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rabi_01_fit.pdf")

plt.show()


## Residual analysis

Residuals are treated as measurable structure unless demonstrated random.

In [ ]:
res = residuals(rabi_model, t, y_obs, params)
rmse = np.sqrt(np.mean(res**2))
mean_res = np.mean(res)
std_res = np.std(res)

print(f"Residual mean = {mean_res:.6f}")
print(f"Residual std  = {std_res:.6f}")
print(f"Fit RMSE      = {rmse:.6f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.plot(t, res, marker="o", markersize=3, linewidth=1)
plt.xlabel("time / pulse duration")
plt.ylabel("residual: observed - fit")
plt.title("Rabi calibration residuals")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rabi_02_residuals.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rabi_02_residuals.pdf")

plt.show()


## Residual autocorrelation

This diagnostic checks whether residuals contain structure over lag rather than behaving like independent random noise.


In [ ]:
def autocorrelation(x):
    x = x - np.mean(x)
    result = np.correlate(x, x, mode="full")
    return result[result.size // 2:]

ac = autocorrelation(res)

plt.figure(figsize=(7, 4))
plt.plot(ac[:50])
plt.title("Residual autocorrelation (structure check)")
plt.xlabel("lag")
plt.ylabel("correlation")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rabi_04_residual_autocorr.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rabi_04_residual_autocorr.pdf")

plt.show()


## CGCS / Triplet-Phase-Lock metric

We compute cosine similarity between observed signal and fitted model.

The working CGCS threshold is:

$$
\cos\theta \geq \frac{1}{\sqrt{1^2+1^2}}
$$

This is the 45° constraint gate.

In [ ]:
metric = phase_lock_metric(y_obs, y_fit)
threshold = 1 / np.sqrt(2)
locked = is_phase_locked(metric, threshold=threshold)
angle_deg = np.degrees(np.arccos(np.clip(metric, -1, 1)))

print(f"CGCS phase-lock metric = {metric:.6f}")
print(f"CGCS threshold         = {threshold:.6f}")
print(f"Angle                  = {angle_deg:.3f} degrees")
print(f"Phase locked?          = {locked}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(["fit alignment", "45° threshold"], [metric, threshold])
plt.ylim(0, 1.05)
plt.ylabel("cosine similarity")
plt.title("CGCS phase-lock check")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/rabi_03_phase_lock.png", dpi=300)
plt.savefig(f"{FIG_DIR}/rabi_03_phase_lock.pdf")

plt.show()


## Calibration summary

In [ ]:
summary = {
    "A_fit": float(A_fit),
    "Omega_fit": float(Omega_fit),
    "B_fit": float(B_fit),
    "residual_rmse": float(rmse),
    "phase_lock_metric": float(metric),
    "phase_lock_threshold": float(threshold),
    "phase_lock_angle_degrees": float(angle_deg),
    "phase_locked": bool(locked),
}

summary

## Save calibration summary


In [ ]:
with open(f"{RESULTS_DIR}/rabi_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/rabi_summary.json")


## Next steps

This notebook can extend directly into:

- `calibration/ramsey/` for detuning and dephasing extraction,
- `calibration/rb/` for gate-fidelity decay,
- `calibration/drift/` for stability over time,
- `noise-mitigation/` for structured residual modeling,
- `control-stack/` for pulse tuning driven by measured calibration.

Guiding rule:

> Start where physics shows up.